In [ ]:
import digitalhub as dh

project = dh.get_or_create_project("demo-guardrails")

# 1. Deploy the service to protect

We will use Python runtime to deploy a simple API to be protected by guardrails. The api makes devision operation given the input data.

In [ ]:
from pathlib import Path
Path("src").mkdir(exist_ok=True)

In [ ]:
%%writefile "src/divisor.py"

import nuclio_sdk
import os
import json

def handler_serve(context: nuclio_sdk.Context, event: nuclio_sdk.Event):
    if isinstance(event.body, bytes):
        body = json.loads(event.body)
    else:
        body = event.body
    
    divident = body['divident']
    divisor = body['divisor']

    return {"result": divident/divisor}

In [ ]:
func = project.new_function(name="divisor",
                            kind="python",
                            python_version="PYTHON3_10",
                            code_src="src/divisor.py",
                            handler="handler_serve"
                           )

In [ ]:
run = func.run(action="serve")

In [ ]:
import requests

data = {
    'divident': 42,
    'divisor': 6
}
res = requests.post(f"http://{run.refresh().status.service['url']}", json=data)
res.json()

# 2. Deploy the guardrail

In [ ]:
%%writefile "src/divisor_guardrail_service.py"

import nuclio_sdk
import os
import json

def handler_serve(context: nuclio_sdk.Context, event: nuclio_sdk.Event):
    if isinstance(event.body, bytes):
        body = json.loads(event.body)
    else:
        body = event.body
    
    divident = body['divident'] if 'divident' in body else 0 
    divisor = body['divisor'] if 'divisor' in body else 0

    if divident and divisor and divisor > 0:
        return event.body

    return context.Response(body="Invalid input data",
                headers={},
                content_type='text/plain',
                status_code=400)

In [ ]:
guardrail_func = project.new_function(name="divisor-guardrail",
                                        kind="guardrail",
                                        python_version="PYTHON3_10",
                                        code_src="src/divisor_guardrail_service.py",
                                        handler="handler_serve",
                                        processing_mode="preprocessor"
                                       )

In [ ]:
guardrail_run = guardrail_func.run(action="serve")

In [ ]:
guardrail_url = guardrail_run.refresh().status.service['url']
guardrail_url

# 3. Expose Protected Service via Service Gateway

In this scenario we expect the platform is equipped with the Envoy Gateway infrastructure configured for guardrails management. 
To protect the service instance with guardrails, we rely on the corresponding gateway and use Envoy Gateway extension for the runs.

Specifically, when the service enable the extension, we obtain:

- the service exposed also behind the preconfigured service Envoy gateway (see the gatewayInfo in service status);
- if the guardrails are configured, the gateway controls the traffic using the ExtProc extension that interacts with the guardrails to implement pre/post processing logic. 

In [ ]:
run = func.run(action="serve", extensions=[{
    "kind": "envoygw",
    "name": "gw",
    "spec": {
        "guardrails": [guardrail_url]
    }
}])

In [ ]:
run.status.gatewayInfo

In [ ]:
PROTECTED_ENDPOINT = f"http://{run.status.gatewayInfo['gatewayEndpoint']}/{run.id}"

In [ ]:
import requests

data = {
    'divident': 42,
    'divisor': 6
}
res = requests.post(PROTECTED_ENDPOINT, json=data)
res.text

In [ ]:
import requests

data = {
    'divident': 42,
    'divisor': 0
}
res = requests.post(PROTECTED_ENDPOINT, json=data)
res.text